# ប្តូរសំឡេង — Amphion DiffWaveNetSVC (ដូច `lucataco/singing_voice_conversion`)

Model នេះជារបស់ **Amphion** (open source, MIT) ហើយ `lucataco/singing_voice_conversion` គ្រាន់តែជា Cog wrapper
របស់ HF Space `amphion/singing_voice_conversion`។ ដូច្នេះយើងទាញយក repo របស់ Space មករត់ក្នុង Colab ផ្ទាល់ —
**ឥតគិតថ្លៃ ឥតកំណត់ចំនួនដង ឥតត្រូវការ API key**។

**ត្រូវការ៖** Runtime → Change runtime type → **T4 GPU**

| | |
|---|---|
| រត់លើ | T4 (free) ~ ១–៣ ដងនៃរយៈពេលសំឡេង |
| ទំហំទាញយក | ~៥ GB (ដំណើរការតែម្តង) |
| Input | សំឡេងច្រៀង **ដោយគ្មានភ្លេង** (vocal stem) |
| Output | សំឡេងដើមដែលប្តូរទៅអ្នកច្រៀងគោលដៅ |

កំណត់សម្គាល់តែមួយ៖ Colab ឥតគិតថ្លៃមានកំណត់ម៉ោង GPU ក្នុងមួយថ្ងៃ (មិនមែនកំណត់ចំនួនដងទេ) — បើអស់ សូមរង់ចាំ ឬប្រើ account ផ្សេង។

## ១. ពិនិត្យ GPU និងដំឡើង library (~៦ នាទី, រត់ម្តង)

In [ ]:
!nvidia-smi -L || echo "❌ គ្មាន GPU — Runtime → Change runtime type → T4 GPU"
%cd /content
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null 2>&1
!pip -q install "setuptools<70" ruamel.yaml tqdm colorama easydict tabulate loguru json5 Cython unidecode inflect librosa typeguard einops omegaconf hydra-core humanfriendly praat-parselmouth pedalboard ffmpeg-python pyworld diffsptk nnAudio torchmetrics pymcd encodec "transformers==4.30.2" "diffusers==0.19.3" accelerate "huggingface_hub[hf_transfer]"
# ContentVec ត្រូវការ fairseq; បើ build មិនចេញ ប្រើ fork ដែលគាំទ្រ Python ថ្មី
!pip -q install fairseq || pip -q install git+https://github.com/One-sixth/fairseq.git
print("✅ ដំឡើងរួច")

## ២. ទាញយក model + weights (~៥ GB, រត់ម្តង)

In [ ]:
import os
from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
ROOT = "/content/Amphion"

if not os.path.exists(os.path.join(ROOT, "inference.py")):
    snapshot_download(
        repo_id="amphion/singing_voice_conversion",
        repo_type="space",
        local_dir=ROOT,
        allow_patterns=[
            "*.py", "*.json", "*.yaml", "*.yml", "*.txt",
            "ckpts/**", "pretrained/**", "models/**", "processors/**",
            "modules/**", "utils/**", "text/**", "egs/**", "config/**",
            "optimizer/**", "preprocessors/**", "examples/*.wav",
        ],
        ignore_patterns=["*.md", "*.gitattributes"],
    )
print("✅ root:", ROOT)
print("ckpt:", os.listdir(f"{ROOT}/ckpts/svc"))
print("pretrained:", os.listdir(f"{ROOT}/pretrained"))

## ៣. បម្លែងសំឡេង

ដាក់ឯកសារសំឡេងរបស់អ្នក (wav/mp3 **ដោយគ្មានភ្លេង**)។ បើទុកប្រអប់ទំនេរ វានឹងបើកផ្ទាំង upload។

In [ ]:
SOURCE = ""  #@param {type:"string"}
TARGET_SINGER = "Taylor Swift"  #@param ["Adele", "John Mayer", "Bruno Mars", "Beyonce", "Michael Jackson", "Taylor Swift", "David Tao 陶喆", "Eason Chan 陈奕迅", "Feng Wang 汪峰", "Jian Li 李健", "Ying Na 那英", "Yijie Shi 石倚洁", "Jacky Cheung 张学友", "Faye Wong 王菲", "Tsai Chin 蔡琴"]
PITCH = "Auto Shift"  #@param ["Auto Shift", "Key Shift"]
KEY_SHIFT = 0  #@param {type:"slider", min:-6, max:6, step:1}
STEPS = 200  #@param {type:"slider", min:50, max:1000, step:50}

import os, shutil, subprocess

SINGER_IDS = {
    "Adele": "vocalist_l1_Adele", "Beyonce": "vocalist_l1_Beyonce",
    "Bruno Mars": "vocalist_l1_BrunoMars", "John Mayer": "vocalist_l1_JohnMayer",
    "Michael Jackson": "vocalist_l1_MichaelJackson", "Taylor Swift": "vocalist_l1_TaylorSwift",
    "Jacky Cheung 张学友": "vocalist_l1_张学友", "Jian Li 李健": "vocalist_l1_李健",
    "Feng Wang 汪峰": "vocalist_l1_汪峰", "Faye Wong 王菲": "vocalist_l1_王菲",
    "Yijie Shi 石倚洁": "vocalist_l1_石倚洁", "Tsai Chin 蔡琴": "vocalist_l1_蔡琴",
    "Ying Na 那英": "vocalist_l1_那英", "Eason Chan 陈奕迅": "vocalist_l1_陈奕迅",
    "David Tao 陶喆": "vocalist_l1_陶喆",
}

if not SOURCE or not os.path.exists(SOURCE):
    from google.colab import files
    ups = files.upload()
    SOURCE = "/content/" + next(iter(ups))

# បម្លែងជា 44.1kHz ម៉ូណូ និងដាក់ក្នុង folder ដោយឡែក (វាបម្លែងគ្រប់ឯកសារក្នុង folder)
IN_DIR = "/content/input"
shutil.rmtree(IN_DIR, ignore_errors=True)
os.makedirs(IN_DIR, exist_ok=True)
name = os.path.splitext(os.path.basename(SOURCE))[0].replace(" ", "_")
prepared = f"{IN_DIR}/{name}.wav"
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", SOURCE, "-ac", "1", "-ar", "44100", prepared], check=True)
print("input:", prepared)

In [ ]:
import os, sys, shutil
sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ["WORK_DIR"] = "./"

import inference

shutil.rmtree("result", ignore_errors=True)
inference.main([
    "--config", "ckpts/svc/vocalist_l1_contentvec+whisper/args.json",
    "--acoustics_dir", "ckpts/svc/vocalist_l1_contentvec+whisper",
    "--vocoder_dir", "pretrained/bigvgan",
    "--target_singer", SINGER_IDS[TARGET_SINGER],
    "--trans_key", "autoshift" if PITCH == "Auto Shift" else str(KEY_SHIFT),
    "--diffusion_inference_steps", str(STEPS),
    "--source", "/content/input",
    "--output_dir", "result",
    "--log_level", "debug",
])

out = f"/content/Amphion/result/{name}/{name}_{SINGER_IDS[TARGET_SINGER]}.wav"
print("✅", out, os.path.getsize(out), "bytes")
from IPython.display import Audio, display
display(Audio(out))

## ៤. ទាញយកលទ្ធផល

បើចង់បាន MP3 តូច និងរក្សាទុកក្នុង Google Drive៖

In [ ]:
import subprocess
mp3 = out.replace(".wav", ".mp3")
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", out, "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)

# (ស្រេចចិត្ត) រក្សាទុកក្នុង Drive
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copy(mp3, "/content/drive/MyDrive/")

from google.colab import files
files.download(mp3)

## ដោះស្រាយបញ្ហា

| បញ្ហា | ដំណោះស្រាយ |
|---|---|
| `pip install fairseq` បរាជ័យ | ស្គ្រីបព្យាយាម fork `One-sixth/fairseq` ស្វ័យប្រវត្តិ — បើនៅតែខុស សូមប្តូរ runtime ទៅ Python 3.10 |
| `CUDA out of memory` | បន្ថយ `STEPS` ទៅ 100–150 ឬកាត់សំឡេងខ្លីជាង ២ នាទី |
| សំឡេងចេញញ័រ/ខូច | input ត្រូវជា **vocal សុទ្ធ** (ឥតភ្លេង) — ប្រើ stem `vocals` ពី site |
| លឿនពេក/យឺតពេក | ប្តូរ `PITCH` → `Key Shift` រួចលៃ `KEY_SHIFT` (± សេមីតូន) |
| Colab បិទ session | ទាញយកលទ្ធផលភ្លាម; weights ត្រូវទាញម្តងទៀតបើ session ថ្មី (ឬ mount Drive) |

**ចំណាំភាសា៖** model នេះបម្លែង *timbre* តែមិនបកប្រែភាសាទេ — វាច្រៀងពាក្យខ្មែរដដែល ដោយសំឡេងអ្នកច្រៀងគោលដៅ។
ដូច្នេះលំដាប់ត្រឹមត្រូវគឺ៖ បកប្រែជាខ្មែរលើ site → យក stem `khmer-speech` → មកបម្លែងនៅទីនេះ។